# 🌲 PHẦN 3: HỌC CÓ GIÁM SÁT (SUPERVISED CLASSIFICATION)
### Mục tiêu: Nhận diện chính xác 5 nhóm hành vi mạng bằng Random Forest
---
### 1. Công thức Tiền xử lý & Chống rò rỉ dữ liệu (Anti-Data Leakage):
- **Biến đổi Log1p** (nén các cột lệch phải `duration`, `src_bytes`, `dst_bytes`):
  $$x' = \ln(1 + \max(0, x))$$
- **Chuẩn hóa Z-Score (`StandardScaler`):** Chỉ tính trung bình $\mu_{\text{train}}$ và độ lệch chuẩn $\sigma_{\text{train}}$ trên tập **Train**, sau đó áp đúng "thước đo của quá khứ" lên tập **Test** để chống rò rỉ dữ liệu:
  $$z_{\text{train}} = \frac{x_{\text{train}} - \mu_{\text{train}}}{\sigma_{\text{train}}}, \qquad z_{\text{test}} = \frac{x_{\text{test}} - \mu_{\text{train}}}{\sigma_{\text{train}}}$$
- **Mã hóa `OneHotEncoder(handle_unknown="ignore")`:** Biến mỗi giá trị chữ thuộc $C$ danh mục thành vector nhị phân $\mathbf{v} \in \{0, 1\}^C$ (học từ điển từ Train; nếu Test gặp giá trị lạ thì điền vector `0` an toàn).


In [2]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
TRAIN_FILE = DATA_DIR / "KDDTrain+.txt"
TEST_FILE = DATA_DIR / "KDDTest+.txt"
if not TRAIN_FILE.exists():
    TRAIN_FILE = Path(r"D:\data_miner\archive\KDDTrain+.txt")
    TEST_FILE = Path(r"D:\data_miner\archive\KDDTest+.txt")

COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag']
SKEWED_COLS = ['duration', 'src_bytes', 'dst_bytes']
NUMERIC_COLS = [c for c in COLUMN_NAMES if c not in CATEGORICAL_COLS + ['label', 'difficulty']]

ATTACK_MAP = {
    'normal': 'Normal',
    'neptune': 'DoS', 'smurf': 'DoS', 'pod': 'DoS', 'teardrop': 'DoS', 'land': 'DoS', 'back': 'DoS',
    'apache2': 'DoS', 'udpstorm': 'DoS', 'processtable': 'DoS', 'mailbomb': 'DoS',
    'portsweep': 'Probe', 'ipsweep': 'Probe', 'nmap': 'Probe', 'satan': 'Probe', 'saint': 'Probe', 'mscan': 'Probe',
    'guess_passwd': 'R2L', 'ftp_write': 'R2L', 'imap': 'R2L', 'phf': 'R2L', 'multihop': 'R2L',
    'warezmaster': 'R2L', 'warezclient': 'R2L', 'spy': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L',
    'snmpguess': 'R2L', 'snmpgetattack': 'R2L', 'httptunnel': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R', 'ps': 'U2R', 'xterm': 'U2R', 'sqlattack': 'U2R'
}

def load_and_clean(file_path):
    df = pd.read_csv(file_path, names=COLUMN_NAMES, header=None)
    for c in CATEGORICAL_COLS:
        df[c] = df[c].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip().str.lower()
    df['attack_cat'] = df['label'].map(lambda x: ATTACK_MAP.get(x, 'Normal'))
    return df

def prepare_numeric(df):
    df_num = df[NUMERIC_COLS].copy()
    for col in SKEWED_COLS:
        df_num[col] = np.log1p(np.maximum(0, df_num[col].astype(np.float32)))
    return df_num.values

df_train = load_and_clean(TRAIN_FILE)
df_test = load_and_clean(TEST_FILE)

# Fit scaler & encoder CHỈ TRÊN TẬP TRAIN (Chống rò rỉ dữ liệu Data Leakage)
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train = np.hstack([scaler.fit_transform(prepare_numeric(df_train)), encoder.fit_transform(df_train[CATEGORICAL_COLS])]).astype(np.float32)
y_train = df_train['attack_cat'].values

X_test = np.hstack([scaler.transform(prepare_numeric(df_test)), encoder.transform(df_test[CATEGORICAL_COLS])]).astype(np.float32)
y_test = df_test['attack_cat'].values

feature_names = NUMERIC_COLS + list(encoder.get_feature_names_out(CATEGORICAL_COLS))
print(f"✅ Đã nạp & tiền xử lý xong! Train: {X_train.shape} | Test độc lập: {X_test.shape}")


### 2. Huấn luyện Random Forest và Công thức đánh giá trên `KDDTest+`
- **Tiêu chuẩn rẽ nhánh cây quyết định (Gini Impurity):** Tại nút $S$ có tỷ lệ lớp $p_k$, tìm điểm cắt tối đa hóa độ lợi Gini Gain $\Delta I_G$:
  $$I_G(S) = 1 - \sum_{k=1}^{K} p_k^2, \qquad \Delta I_G = I_G(S) - \left(\frac{|S_L|}{|S|} I_G(S_L) + \frac{|S_R|}{|S|} I_G(S_R)\right)$$
- **Bỏ phiếu đa số (Bagging Ensemble với $B = 50$ cây, `max_depth=12`):**
  $$\hat{y} = \arg\max_{c} \sum_{b=1}^{B} \mathbb{I}\big(T_b(\mathbf{x}) = c\big)$$
- **Các chỉ số chấm điểm trên tập kiểm thử độc lập `KDDTest+`:**
  $$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}, \qquad \text{Precision} = \frac{TP}{TP + FP}, \qquad \text{Recall} = \frac{TP}{TP + FN}, \qquad F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$


In [2]:
rf = RandomForestClassifier(n_estimators=50, max_depth=12, random_state=42, n_jobs=1)
rf.fit(X_train, y_train)

train_acc = rf.score(X_train, y_train) * 100
test_acc = rf.score(X_test, y_test) * 100
y_pred = rf.predict(X_test)

print(f"✅ Độ chính xác trên tập Train (KDDTrain+): {train_acc:.2f}%")
print(f"✅ Độ chính xác trên tập Test độc lập (KDDTest+): {test_acc:.2f}%\n")
print("Báo cáo phân loại chi tiết trên tập Test độc lập (Classification Report):")
print(classification_report(y_test, y_pred, zero_division=0))


### 3. Trực quan hóa Ma trận nhầm lẫn & Trọng số đặc trưng
- **Ma trận nhầm lẫn (Confusion Matrix):** Phần tử $CM_{i,j}$ đếm số mẫu có nhãn thật là lớp $i$ nhưng mô hình dự đoán là lớp $j$ (các số trên **đường chéo chính $CM_{i,i}$** là số dự đoán đúng).
- **Độ quan trọng của đặc trưng (Feature Importance - Mean Decrease in Impurity):** Trung bình mức giảm độ vẩn đục Gini mà đặc trưng $f$ đóng góp trên toàn bộ $B = 50$ cây:
  $$\text{Importance}(f) = \frac{1}{B} \sum_{b=1}^{B} \sum_{t \in T_b:\, \text{split}=f} \frac{N_t}{N} \Delta I_G(t)$$


In [3]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5), dpi=120)

# 1. Confusion Matrix trên tập Test độc lập (KDDTest+)
labels = sorted(list(set(y_test)))
cm = confusion_matrix(y_test, y_pred, labels=labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax[0])
ax[0].set_title("Ma Trận Nhầm Lẫn Random Forest 5 Lớp (KDDTest+)", fontweight="bold")
ax[0].set_xlabel("Nhãn dự đoán")
ax[0].set_ylabel("Nhãn thực tế")

# 2. Top 10 Đặc Trưng Quan Trọng Nhất (Hiển thị đúng tên cột)
top_feat = pd.Series(rf.feature_importances_, index=feature_names).nlargest(10).sort_values(ascending=True)
ax[1].barh(top_feat.index, top_feat.values, color="#16a085", edgecolor="black", alpha=0.85)
ax[1].set_title("Top 10 Đặc Trưng Quan Trọng Nhất (Gini Importance)", fontweight="bold")
ax[1].set_xlabel("Trọng số phân nhánh")

plt.tight_layout()
plt.show()
